# 07.04 — Temporal Alignment

Confirm that information at forecast origin `t` points to exactly `t+h` for the selected target horizon.

In [1]:
# Import libraries
from pathlib import Path
import sys

In [2]:
# Define the root directory of the project
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = PROJECT_ROOT / 'configs' / 'target_definition.yaml'
CONFIG_PATH

WindowsPath('e:/jcuenca/OneDrive - GUSCanada/5toTerm/01_Capstone/Codigo/ontario-electricity-peak-risk/configs/target_definition.yaml')

In [4]:
# Import module to manage the Target Definition process
from src.ontario_peak_risk.target_definition.common import (
    load_target_config,
    load_feature_dataset,
    ensure_directories,
)

In [5]:
# Configure the Target Definition process
CONFIG, _ = load_target_config(CONFIG_PATH)
OUTPUT_DIR, REPORTS_DIR, DOCS_DIR = ensure_directories(CONFIG, PROJECT_ROOT)
feature_dataset = load_feature_dataset(CONFIG, PROJECT_ROOT)
feature_dataset.shape

(262944, 100)

In [6]:
# Import module to build the Peak Risk Target matrix and generate reports
from src.ontario_peak_risk.target_definition.forecasting_target import build_forecasting_target_matrix
from src.ontario_peak_risk.target_definition.alignment import build_alignment_sample

In [7]:
# Build the Forecasting Target matrix
FORECAST_CFG = CONFIG['target_definition']['forecasting']
forecast_matrix = build_forecasting_target_matrix(
    feature_dataset,
    target_column=FORECAST_CFG['target_column'],
    horizon_hours=FORECAST_CFG['horizon_hours'],
    target_prefix=FORECAST_CFG['target_prefix'],
)

forecast_matrix.shape

(262944, 28)

In [8]:
# Display the first few rows of the forecasting target matrix
forecast_matrix.head()

,fsa,forecast_origin,target_h01,target_h02,target_h03,target_h04,target_h05,target_h06,target_h07,target_h08,...,target_h17,target_h18,target_h19,target_h20,target_h21,target_h22,target_h23,target_h24,available_horizons,complete_24h_target
0,L4T,2021-01-01 00:00:00,9585.1,9015.5,8593.1,8353.5,8334.4,8499.5,8784.2,9313.2,...,13737.4,13772.0,13490.4,13255.0,12622.0,11607.1,10571.4,9630.6,24,1
1,L4T,2021-01-01 01:00:00,9015.5,8593.1,8353.5,8334.4,8499.5,8784.2,9313.2,10062.2,...,13772.0,13490.4,13255.0,12622.0,11607.1,10571.4,9630.6,8984.5,24,1
2,L4T,2021-01-01 02:00:00,8593.1,8353.5,8334.4,8499.5,8784.2,9313.2,10062.2,10968.0,...,13490.4,13255.0,12622.0,11607.1,10571.4,9630.6,8984.5,8434.7,24,1
3,L4T,2021-01-01 03:00:00,8353.5,8334.4,8499.5,8784.2,9313.2,10062.2,10968.0,11971.4,...,13255.0,12622.0,11607.1,10571.4,9630.6,8984.5,8434.7,8151.9,24,1
4,L4T,2021-01-01 04:00:00,8334.4,8499.5,8784.2,9313.2,10062.2,10968.0,11971.4,12425.2,...,12622.0,11607.1,10571.4,9630.6,8984.5,8434.7,8151.9,8059.8,24,1


In [9]:
# Build the alignment sample
alignment_sample = build_alignment_sample(
    feature_dataset,
    forecast_matrix,
    target_column=FORECAST_CFG['target_column'],
)
alignment_sample.shape

(54, 7)

In [10]:
# Display the distribution of matches and mismatches in the alignment sample
alignment_sample

,fsa,forecast_origin,horizon_hours,target_timestamp,target_matrix_value,source_value,match
0,L4T,2021-01-01 00:00:00,1,2021-01-01 01:00:00,9585.1,9585.1,True
1,L4T,2021-01-01 00:00:00,2,2021-01-01 02:00:00,9015.5,9015.5,True
2,L4T,2021-01-01 00:00:00,24,2021-01-02 00:00:00,9630.6,9630.6,True
3,L4T,2021-01-01 01:00:00,1,2021-01-01 02:00:00,9015.5,9015.5,True
4,L4T,2021-01-01 01:00:00,2,2021-01-01 03:00:00,8593.1,8593.1,True
5,L4T,2021-01-01 01:00:00,24,2021-01-02 01:00:00,8984.5,8984.5,True
6,L4T,2021-01-01 02:00:00,1,2021-01-01 03:00:00,8593.1,8593.1,True
7,L4T,2021-01-01 02:00:00,2,2021-01-01 04:00:00,8353.5,8353.5,True
8,L4T,2021-01-01 02:00:00,24,2021-01-02 02:00:00,8434.7,8434.7,True
9,M5R,2021-01-01 00:00:00,1,2021-01-01 01:00:00,7124.9,7124.9,True


In [11]:
# Display the distribution of matches and mismatches in the alignment sample
alignment_sample['match'].value_counts(dropna=False)

match
True    54
Name: count, dtype: int64